# 05 Map-Based Visibility Reference

This notebook is the structured comparison track: derive reliability from an occupancy map instead of hand-designing `q(x)` directly.

Demonstration question:
- If we assume a map and a camera location, can we derive a visibility field that is interpretable enough to use as a comparison baseline?
- How does a hard line-of-sight rule differ from a softened occupancy-ray model?
- Does the planner react differently when the visibility model comes from environment structure rather than an oracle field?


## Visibility Models, Formulas, And Tradeoffs

Hard visibility model:

```math
q_{\mathrm{hard}}(x) =
\begin{cases}
1, & \text{if the ray from camera to } x \text{ avoids occupied cells} \\
0, & \text{otherwise}
\end{cases}
```

Soft occupancy-ray model:

```math
q_{\mathrm{soft}}(x) = \exp\left(-\tau \; \overline{\mathrm{occ}}\_{\text{ray}}\right)
```

where `overline{occ}_ray` is the mean occupancy sampled along the camera-to-state ray.

Why use these two models:
- hard LoS is maximally interpretable,
- soft occupancy-ray is still cheap, but avoids brittle binary transitions.

Limitations:
- this is still not a full inverse sensor model,
- the map is hand-constructed here,
- no uncertainty over the map itself is represented.

Natural alternatives:
- occupancy-grid update from actual simulator or sensor data,
- GP occupancy map,
- direct learned `q(x)` without map assumptions.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "scripts").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import numpy as np
import matplotlib.pyplot as plt

from scripts.state_dependent_observation_helpers import (
    covariance_logdet_series,
    covariance_trace_series,
    evaluate_visibility_on_grid,
    grid_states,
    hard_line_of_sight_q,
    make_action_library,
    make_default_camera,
    make_observation_fn,
    make_occupancy_grid,
    observation_covariance,
    process_covariance_from_rho,
    raycast_visibility_q,
    score_action_library,
    simulate_receding_horizon,
)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

dt = 0.2
rho = np.array([1e-2, 1e-2], dtype=float)
risk_weight = 1.0
ambiguity_weight = 1.0
control_weight = 0.10

camera = make_default_camera()
camera_xy = camera.cam_pos[:2]
g = make_observation_fn(camera, obs_mode="uv")
Q = process_covariance_from_rho(dt=dt, rho_xy=rho[0])
R_good = observation_covariance(obs_mode="uv", uv_std=2.5)
R_bad = observation_covariance(obs_mode="uv", uv_std=20.0)
actions = make_action_library(v_values=(0.0, 0.22, 0.35, 0.45), w_values=(-1.2, -0.8, -0.35, 0.0, 0.35, 0.8, 1.2))

occ_grid = make_occupancy_grid(
    xmin=-5.0,
    xmax=5.0,
    ymin=-5.0,
    ymax=1.0,
    resolution=0.10,
    rectangles=[(-0.75, 0.75, -2.60, -1.20)],
    circles=[(0.0, -3.7, 0.35)],
    border_occupancy=True,
)

start_xy = np.array([-0.8, -2.0], dtype=float)
goal_xy = np.array([0.8, -1.5], dtype=float)
theta0 = 0.0
mean0 = np.array([start_xy[0], start_xy[1], theta0], dtype=float)
cov0 = np.diag([0.30, 0.30, 0.12])
goal_state = np.array([goal_xy[0], goal_xy[1], theta0], dtype=float)
goal_obs = g(goal_state)
goal_obs_cov = observation_covariance(obs_mode="uv", uv_std=24.0)

hard_q_fn = lambda s: hard_line_of_sight_q(s, occ_grid, camera_xy, threshold=0.35, n_samples=120)
soft_q_fn = lambda s: raycast_visibility_q(s, occ_grid, camera_xy, tau=8.0, n_samples=120)


In [ ]:
xmin, xmax, ymin, ymax = occ_grid.extent
xs, ys, states = grid_states(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, nx=91, ny=61, theta=theta0)
hard_q_map = evaluate_visibility_on_grid(states, hard_q_fn)
soft_q_map = evaluate_visibility_on_grid(states, soft_q_fn)

sample_states = [
    np.array([0.0, -1.5, theta0]),
    np.array([0.0, -2.0, theta0]),
    np.array([0.0, -3.0, theta0]),
    np.array([1.0, -2.0, theta0]),
]
for state in sample_states:
    print(
        f"state={state[:2]}  hard_q={hard_q_fn(state):.3f}  soft_q={soft_q_fn(state):.3f}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6.0), constrained_layout=True)
ax.imshow(occ_grid.occupancy, origin="lower", extent=occ_grid.extent, cmap="Greys")
ax.scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="tab:blue", label="camera")

for idx, state in enumerate(sample_states):
    q_soft = soft_q_fn(state)
    q_hard = hard_q_fn(state)
    ax.plot(
        [camera_xy[0], state[0]],
        [camera_xy[1], state[1]],
        linewidth=2,
        label=f"ray {idx + 1}: soft={q_soft:.2f}, hard={q_hard:.0f}",
    )
    ax.scatter(state[0], state[1], s=55, color="tab:red")

ax.set_title("Ray-based interpretation of the hard and soft visibility models")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.axis("equal")
ax.legend(loc="best")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].imshow(occ_grid.occupancy, origin="lower", extent=occ_grid.extent, cmap="Greys")
axes[0].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="tab:blue", label="camera")
axes[0].scatter(mean0[0], mean0[1], marker="o", s=70, color="tab:green", label="start")
axes[0].scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")
axes[0].set_title("Occupancy map")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].legend(loc="best")
axes[0].axis("equal")

axes[1].imshow(hard_q_map, origin="lower", extent=occ_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis")
axes[1].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white")
axes[1].set_title("Hard line-of-sight q(x)")
axes[1].set_xlabel("x [m]")
axes[1].set_ylabel("y [m]")
axes[1].axis("equal")

axes[2].imshow(soft_q_map, origin="lower", extent=occ_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis")
axes[2].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white")
axes[2].set_title("Soft occupancy-ray q(x)")
axes[2].set_xlabel("x [m]")
axes[2].set_ylabel("y [m]")
axes[2].axis("equal")

plt.show()


## Planner Comparison Guide

There are three planner views here:
- constant `R`: ignores visibility structure,
- soft occupancy `q(x)`: visibility fades smoothly with occlusion along the ray,
- hard LoS `q(x)`: visibility flips abruptly between 0 and 1.

What to watch for:
- whether the first selected control changes,
- whether the resulting trajectory spends less time in low-visibility regions,
- whether hard visibility is too brittle compared with the soft reference.


In [ ]:
constant_candidates = score_action_library(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_good,
    q_fn=None,
    horizon=8,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)

soft_candidates = score_action_library(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_bad,
    q_fn=soft_q_fn,
    horizon=8,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)

hard_candidates = score_action_library(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_bad,
    q_fn=hard_q_fn,
    horizon=8,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)

for label, item in [("constant", constant_candidates[0]), ("soft", soft_candidates[0]), ("hard", hard_candidates[0])]:
    print(
        f"{label:>8}: u={item.control}, total={item.total:.3f}, min_q={item.min_q:.3f}, "
        f"risk_w={item.risk_term:.3f}, amb_w={item.ambiguity_term:.3f}"
    )

constant_run = simulate_receding_horizon(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_good,
    q_fn=None,
    horizon=8,
    n_steps=18,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)

soft_run = simulate_receding_horizon(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_bad,
    q_fn=soft_q_fn,
    horizon=8,
    n_steps=18,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)

hard_run = simulate_receding_horizon(
    mean0,
    cov0,
    actions,
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=goal_obs,
    goal_obs_cov=goal_obs_cov,
    R_good=R_good,
    R_bad=R_bad,
    q_fn=hard_q_fn,
    horizon=8,
    n_steps=18,
    risk_weight=risk_weight,
    ambiguity_weight=ambiguity_weight,
    control_weight=control_weight,
    approx="ET2",
    add_ambiguity=True,
)


In [ ]:
def path_length(states):
    diffs = np.diff(states[:, :2], axis=0)
    return float(np.sum(np.linalg.norm(diffs, axis=1)))

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

axes[0, 0].imshow(soft_q_map, origin="lower", extent=occ_grid.extent, vmin=0.0, vmax=1.0, cmap="viridis", alpha=0.85)
axes[0, 0].imshow(occ_grid.occupancy, origin="lower", extent=occ_grid.extent, cmap="Greys", alpha=0.28)
axes[0, 0].plot(constant_run["means"][:, 0], constant_run["means"][:, 1], label="constant R", linewidth=2)
axes[0, 0].plot(soft_run["means"][:, 0], soft_run["means"][:, 1], label="soft occupancy q(x)", linewidth=2)
axes[0, 0].plot(hard_run["means"][:, 0], hard_run["means"][:, 1], label="hard LoS q(x)", linewidth=2)
axes[0, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=90, color="white", label="camera")
axes[0, 0].scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")
axes[0, 0].set_title("Trajectories on occupancy-derived visibility field")
axes[0, 0].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[0, 0].axis("equal")
axes[0, 0].legend(loc="best")

axes[0, 1].plot(covariance_trace_series(constant_run["covs"]), label="constant R", marker="o")
axes[0, 1].plot(covariance_trace_series(soft_run["covs"]), label="soft occupancy q(x)", marker="o")
axes[0, 1].plot(covariance_trace_series(hard_run["covs"]), label="hard LoS q(x)", marker="o")
axes[0, 1].set_title("Covariance trace over time")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("trace(S)")
axes[0, 1].legend()

axes[1, 0].plot(soft_run["q_values"], label="soft occupancy q(x)", marker="o")
axes[1, 0].plot(hard_run["q_values"], label="hard LoS q(x)", marker="o")
axes[1, 0].set_title("Encountered q(x) along executed path")
axes[1, 0].set_xlabel("step")
axes[1, 0].set_ylabel("q(x)")
axes[1, 0].legend()

axes[1, 1].plot(covariance_logdet_series(constant_run["covs"]), label="constant R", marker="o")
axes[1, 1].plot(covariance_logdet_series(soft_run["covs"]), label="soft occupancy q(x)", marker="o")
axes[1, 1].plot(covariance_logdet_series(hard_run["covs"]), label="hard LoS q(x)", marker="o")
axes[1, 1].set_title("Covariance logdet over time")
axes[1, 1].set_xlabel("step")
axes[1, 1].set_ylabel("log det(S)")
axes[1, 1].legend()

plt.show()

for label, run in [("constant", constant_run), ("soft", soft_run), ("hard", hard_run)]:
    q_stats = 1.0 if run["q_values"].size == 0 else float(np.mean(run["q_values"]))
    q_min = 1.0 if run["q_values"].size == 0 else float(np.min(run["q_values"]))
    print(
        f"{label:>8}: path_length={path_length(run['means']):.3f}, "
        f"mean_q={q_stats:.3f}, min_q={q_min:.3f}, final_trace={np.trace(run['covs'][-1]):.4f}"
    )


## Advantages, Limitations, Alternatives, Sources

Advantages:
- environment-structured and interpretable,
- easy to compare against synthetic direct `q(x)`,
- naturally motivates future map-learning extensions.

Limitations:
- assumes a known camera pose and map,
- hard LoS can be unrealistically brittle,
- soft occupancy-ray is only a proxy, not a full probabilistic visibility model.

Alternatives:
- GP occupancy maps,
- signed-distance or volumetric visibility models,
- direct learned reliability from logs without explicit map reasoning.

Local anchors:
- [`state_dependent_observation_helpers.py`](state_dependent_observation_helpers.py)
- [`../src/docs/CONTEXT_FOR_AI.txt`](../src/docs/CONTEXT_FOR_AI.txt)
- [`../docs/state_dependent_observation_plan.md`](../docs/state_dependent_observation_plan.md)

Background reading:
- Elfes, *A tesselated probabilistic representation for spatial robot perception and navigation* (1989): https://ntrs.nasa.gov/citations/19900019759
- Elfes, *Using occupancy grids for mobile robot perception and navigation* (1989 overview): https://ouci.dntb.gov.ua/en/works/4b6Qp5v4/
